# Role Generation for Spider Database Tables

This notebook performs role-based access control (RBAC) analysis for the Spider database collection using LLM.

### 0. Import lib and env

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('/home/feiy/Role-SQL-benchmark')
sys.path.append(str(project_root))

# Import required modules
from src.role_parser import RoleGenerator, ParallelRoleGenerator
from src.utils.sql_data_process import SpiderDataProcessor
from src.utils.role_sql_generate import RoleSQLGenerator
from dotenv import load_dotenv
import os
import json
from datetime import datetime
import logging
import random
import importlib
import src.llm_oracle as oracle

# Setup global timestamp for this run
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Setup output directories
log_dir = project_root / 'logs'
output_dir = project_root / 'outputs'

for directory in [log_dir, output_dir]:
    directory.mkdir(exist_ok=True)

# Configure logging
log_file = log_dir / f'role_assignment_{RUN_TIMESTAMP}.log'
logging.getLogger().handlers.clear()

logger = logging.getLogger('role_assignment')
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(str(log_file))
console_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

for handler in [file_handler, console_handler]:
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False
logger.info(f"Starting new session at {RUN_TIMESTAMP}")
logger.info(f"Log file: {log_file}")
logger.info(f"Output directory: {output_dir}")

2025-09-17 16:03:38,388 - INFO - Starting new session at 20250917_160338
2025-09-17 16:03:38,389 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250917_160338.log
2025-09-17 16:03:38,390 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs
2025-09-17 16:03:38,389 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250917_160338.log
2025-09-17 16:03:38,390 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs


/home/feiy/anaconda3/envs/llm4db/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables and API keys
load_dotenv()
importlib.reload(oracle)

<module 'src.llm_oracle' from '/home/feiy/Role-SQL-benchmark/src/llm_oracle/__init__.py'>

In [3]:
# DeepSeek demo
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not DEEPSEEK_API_KEY:
    print("Warning: Cannot find DEEPSEEK_API_KEY in environment")
    print("Please set it in the .env file or environment")
if not OPENAI_API_KEY:
    print("Warning: Cannot find OPENAI_API_KEY in environment")
    print("Please set it in the .env file or environment")


### 1. Setup LLM Oracle instance

In [4]:
def create_oracle(model_name, api_key):
    return oracle.Oracle(model_name, api_key)

#### 1.1. Deepseek test

##### 1.1(a) deepseek demo test

In [5]:
# if not DEEPSEEK_API_KEY:
#     print("Warning: DEEPSEEK_API_KEY not found in environment variables")
#     print("Please set it in your .env file or environment")
# else:
#     # Create Oracle instance with DeepSeek model
#     oracle = Oracle(model="deepseek-chat", apikey=DEEPSEEK_API_KEY)
    
#     # Test the model
#     test_response = oracle.query(
#         prompt_sys="You are a helpful assistant.",
#         prompt_user="Say Hi.",
#         temp=0.7,
#         top_p=0.9
#     )
    
#     print("Model Response:")
#     print("="*50)
#     print(test_response['answer'])

##### 1.1(b) prompt caching test

In [6]:
# # Create two identical requests to test caching
# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""

# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""

In [7]:
# def make_query(prompt_sys, prompt_user):
#     """Execute query and return response"""
#     response = oracle.query(
#         prompt_sys=prompt_sys,
#         prompt_user=prompt_user,
#     )
#     # if response.get('answer'):
#     #     print(f"Response: {response['answer']}")
#     return response

# def print_cache_stats(response, label=None):
#     """Print cache statistics for a response"""
#     if label:
#         print(f"\n=== {label} Statistics ===")
    
#     usage = response.get('usage', {})
#     prompt_details = usage.get('prompt_tokens_details', None)
    
#     print("\nToken Statistics:")
#     print(f"  Prompt tokens: {usage.get('prompt_tokens', 0)}")
#     print(f"  Completion tokens: {usage.get('completion_tokens', 0)}")
#     print(f"  Total tokens: {usage.get('total_tokens', 0)}")
    
#     print("\nCache Statistics:")
#     if prompt_details:
#         if isinstance(prompt_details, str):
#             print(f"  Raw info: {prompt_details}")
#         else:
#             cached = getattr(prompt_details, 'cached_tokens', 0)
#             non_cached = usage.get('prompt_tokens', 0) - cached
#             print(f"  Cached tokens: {cached}")
#             print(f"  Non-cached tokens: {non_cached}")
#             if cached > 0:
#                 print(f"  Cache hit rate: {(cached / usage.get('prompt_tokens', 1)) * 100:.1f}%")

In [8]:
# response1 = make_query(prompt_sys, prompt_user)
# response2 = make_query(prompt_sys, prompt_user)
# response3 = make_query(prompt_sys, prompt_user)

# print_cache_stats(response1, "First Call")
# print_cache_stats(response2, "Second Call")
# print_cache_stats(response3, "Third Call")

#### 1.2. OpenAI Test

In [9]:
# Example: create Oracle instance (model and api_key should be set according to your environment)

# MODEL_NAME = 'gpt-4o'  # or any supported model
# create_oracle_instance = create_oracle(model_name=MODEL_NAME, api_key=OPENAI_API_KEY)
# print(f"Oracle instance created for model: {MODEL_NAME}")

In [10]:
# build a simple prompt for testing

# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""
# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""
# response = openai_oracle_instance.query(
#     prompt_sys=prompt_sys,
#     prompt_user=prompt_user,
# )
# print(f"Response: {response['answer']}")

### 2. Role Assignment for Spider Database Tables

This section aims to:
1. Read schema information from Spider database
2. Use LLM to generate appropriate roles for each table

#### 2.1 Overview for Spider dataset
This Part will:
1. Basic stats for Spider Database
2. Process and reformat spider file to training dataset

In [11]:
# Initialize processor
processor = SpiderDataProcessor(project_root)

# Get all database folders and statistics
db_folders = processor.get_db_folders()
db_stats = processor.get_db_statistics()

# Calculate and log statistics
logger.info("\nSpider Database Statistics:")
logger.info("-" * 40)

total_dbs = len(db_stats)
dbs_with_sqlite = sum(1 for stats in db_stats.values() if stats['has_sqlite'])
dbs_with_schema = sum(1 for stats in db_stats.values() if stats['has_schema'])
total_tables = sum(stats['table_count'] for stats in db_stats.values())

logger.info(f"Found {len(db_folders)} databases in Spider dataset")
logger.info(f"Total databases in statistics: {total_dbs}")
logger.info(f"Databases with SQLite files: {dbs_with_sqlite}")
logger.info(f"Databases with schema files: {dbs_with_schema}")
logger.info(f"Total tables across all databases: {total_tables}")
logger.info(f"Average tables per database: {total_tables/dbs_with_sqlite:.2f}")

# Additional Spider dataset information
logger.info("\nDetailed database statistics have been saved to spider_info.json")
logger.info("You can find it in the data directory")

# Process Spider train data
logger.info("\nProcessing Spider Training Data:")
logger.info("-" * 40)
try:
    processor.process_spider_train_data()
except Exception as e:
    logger.error(f"Error processing Spider train data: {str(e)}")
    logger.error("Please check if train_spider.json exists and is accessible")

2025-09-17 16:03:38,474 - INFO - 
Spider Database Statistics:
2025-09-17 16:03:38,475 - INFO - ----------------------------------------
2025-09-17 16:03:38,476 - INFO - Found 166 databases in Spider dataset
2025-09-17 16:03:38,477 - INFO - Total databases in statistics: 166
2025-09-17 16:03:38,477 - INFO - Databases with SQLite files: 166
2025-09-17 16:03:38,478 - INFO - Databases with schema files: 148
2025-09-17 16:03:38,479 - INFO - Total tables across all databases: 876
2025-09-17 16:03:38,480 - INFO - Average tables per database: 5.28
2025-09-17 16:03:38,480 - INFO - 
Detailed database statistics have been saved to spider_info.json
2025-09-17 16:03:38,481 - INFO - You can find it in the data directory
2025-09-17 16:03:38,481 - INFO - 
Processing Spider Training Data:
2025-09-17 16:03:38,482 - INFO - ----------------------------------------
2025-09-17 16:03:38,475 - INFO - ----------------------------------------
2025-09-17 16:03:38,476 - INFO - Found 166 databases in Spider datase

#### 2.2 Prompt Design and Role Assignment

The part is designed to (refer to configs/prompts.py for detailed prompt):
1. Provide clear context about the task (Role-Based Access Control)
2. Guide the LLM to analyze table schema and relationships
3. Generate appropriate role names and descriptions
4. Maintain consistency across different tables

In [ ]:
# Process all databases in batches
BATCH_SIZE = 12
N_WORKERS = 10

# Filter databases that have schema files
valid_dbs = [db for db in db_folders if (db / "schema.sql").exists()]
total_dbs = len(valid_dbs)
n_batches = (total_dbs + BATCH_SIZE - 1) // BATCH_SIZE  # Ceiling division

logger.info(f"Starting batch processing for all databases:")
logger.info(f"Total valid databases: {total_dbs}")
logger.info(f"Batch size: {BATCH_SIZE}")
logger.info(f"Number of batches: {n_batches}")
logger.info(f"Workers per batch: {N_WORKERS}")

# Initialize role generator (reused for all batches)
generator = ParallelRoleGenerator(model="deepseek-chat", api_key=DEEPSEEK_API_KEY, n_workers=N_WORKERS)
logger.info(f"Initialized ParallelRoleGenerator with model: deepseek-chat")

# Process each batch
all_role_assignments = {}
total_processed = 0
total_roles = 0

for batch_idx in range(n_batches):
    batch_start = batch_idx * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, total_dbs)
    batch_dbs = valid_dbs[batch_start:batch_end]
    
    logger.info(f"\nProcessing Batch {batch_idx + 1}/{n_batches}")
    logger.info(f"Databases in this batch: {[db.name for db in batch_dbs]}")
    
    # Process batch in parallel
    sqlite_paths = {db.name: str(db / f"{db.name}.sqlite") for db in batch_dbs}
    results = generator.process_databases_parallel(batch_dbs, sqlite_paths=sqlite_paths)
    
    # Process results from this batch
    batch_processed = 0
    batch_roles = 0
    
    for result in results:
        if result and result.get('roles'):
            batch_processed += 1
            roles_count = len(result['roles'])
            batch_roles += roles_count
            all_role_assignments[result['database']] = result['roles']
        else:
            logger.error(f"Failed to process one of the databases in batch {batch_idx + 1}")
    
    # Update totals
    total_processed += batch_processed
    total_roles += batch_roles
    
    # Log batch results
    logger.info(f"Batch {batch_idx + 1} completed:")
    logger.info(f"- Databases processed in this batch: {batch_processed}/{len(batch_dbs)}")
    logger.info(f"- Roles generated in this batch: {batch_roles}")
    logger.info(f"- Total progress: {total_processed}/{total_dbs} databases processed")

# Prepare final metadata
assignments_data = {
    'assignments': all_role_assignments,
    'metadata': {
        'timestamp': RUN_TIMESTAMP,
        'total_databases': total_dbs,
        'processed_databases': total_processed,
        'total_roles_generated': total_roles,
        'batch_size': BATCH_SIZE,
        'n_workers': N_WORKERS,
        'n_batches': n_batches
    }
}

# Save all results
if all_role_assignments:
    output_file = generator.save_assignments_parallel(assignments_data, output_dir, RUN_TIMESTAMP)

logger.info(f"\nAll batches completed:")
logger.info(f"- Total databases processed: {total_processed}/{total_dbs}")
logger.info(f"- Total roles generated: {total_roles}")
logger.info(f"- Average roles per database: {total_roles/total_processed if total_processed else 0:.2f}")

2025-09-17 16:03:38,799 - INFO - Starting role assignment process for 12 databases
2025-09-17 16:03:38,819 - INFO - Initialized ParallelRoleGenerator with model: deepseek-chat
2025-09-17 16:03:38,820 - INFO - Selected databases: ['customers_and_addresses', 'county_public_safety', 'university_basketball', 'movie_1', 'architecture', 'election_representative', 'local_govt_and_lot', 'medicine_enzyme_interaction', 'farm', 'protein_institute', 'epinions_1', 'device']
2025-09-17 16:03:38,821 - INFO - Starting parallel processing of databases
2025-09-17 16:03:38,822 - ERROR - Database 'epinions_1' skipped: schema.sql file not found
2025-09-17 16:03:38,823 - INFO - Processing 11 databases (1 skipped)
2025-09-17 16:03:38,823 - INFO - Skipped databases: epinions_1
2025-09-17 16:03:38,824 - INFO - Starting parallel API calls with 10 workers
2025-09-17 16:03:38,819 - INFO - Initialized ParallelRoleGenerator with model: deepseek-chat
2025-09-17 16:03:38,820 - INFO - Selected databases: ['customers_a

Processing Items: 100%|██████████| 11/11 [00:16<00:00,  1.54s/it]

2025-09-17 16:03:55,732 - INFO - Summary Statistics: Total API Calls: 11, Total Prompt Tokens: 29108, Total Completion Tokens: 1422, Total Cached Tokens: 18240, Overall Cache Hit Rate: 62.7%
2025-09-17 16:03:55,735 - INFO - Generated 4 roles for customers_and_addresses - Cached Tokens: 448; Uncached tokens: 6235; Hit rates: 6.7%
2025-09-17 16:03:55,737 - INFO - Generated 3 roles for county_public_safety - Cached Tokens: 1600; Uncached tokens: 16; Hit rates: 99.0%
2025-09-17 16:03:55,738 - INFO - Generated 3 roles for university_basketball - Cached Tokens: 1088; Uncached tokens: 58; Hit rates: 94.9%
2025-09-17 16:03:55,739 - INFO - Generated 4 roles for movie_1 - Cached Tokens: 448; Uncached tokens: 782; Hit rates: 36.4%
2025-09-17 16:03:55,740 - INFO - Generated 4 roles for architecture - Cached Tokens: 448; Uncached tokens: 1065; Hit rates: 29.6%
2025-09-17 16:03:55,735 - INFO - Generated 4 roles for customers_and_addresses - Cached Tokens: 448; Uncached tokens: 6235; Hit rates: 6.7%


### 3. Generate Role-Based SQL Dataset

Generate role-based text2sql dataset by combining Spider data with role assignments.

In [13]:
# Initialize generator with our project paths
generator = RoleSQLGenerator(
    project_root=project_root,
    spider_train_path=str(project_root / 'data/spider_train_data.json'),
    output_dir=str(project_root / 'outputs')
)

# Use the same timestamp for role assignments file
role_assignments_file = str(project_root / f'outputs/role_assignments_{RUN_TIMESTAMP}.json')

# Generate dataset using our latest role assignments
dataset = generator.generate_role_sql_dataset(role_file_path=role_assignments_file)

# Save dataset with matching timestamp
output_path = str(project_root / f'outputs/role_sql_dataset_{RUN_TIMESTAMP}.json')
generator.save_dataset(dataset, output_path=output_path)

# Print some statistics
total_examples = len(dataset)
databases = len({example['db_id'] for example in dataset})
roles = len({(example['db_id'], example['role']) for example in dataset})
denied_queries = sum(1 for example in dataset if "Sorry, I cannot answer." in example['query'])

logger.info("\nDataset Generation Statistics:")
logger.info("-" * 40)
logger.info(f"Total examples generated: {total_examples}")
logger.info(f"Number of databases: {databases}")
logger.info(f"Number of unique roles: {roles}")
logger.info(f"Queries denied due to permissions: {denied_queries} ({denied_queries/total_examples*100:.2f}%)")
logger.info(f"Output saved to: {output_path}")

2025-09-17 16:03:56,146 - INFO - 
Dataset Generation Statistics:
2025-09-17 16:03:56,147 - INFO - ----------------------------------------
2025-09-17 16:03:56,148 - INFO - Total examples generated: 1791
2025-09-17 16:03:56,149 - INFO - Number of databases: 11
2025-09-17 16:03:56,150 - INFO - Number of unique roles: 42
2025-09-17 16:03:56,150 - INFO - Queries denied due to permissions: 603 (33.67%)
2025-09-17 16:03:56,151 - INFO - Output saved to: /home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_20250917_160338.json
2025-09-17 16:03:56,147 - INFO - ----------------------------------------
2025-09-17 16:03:56,148 - INFO - Total examples generated: 1791
2025-09-17 16:03:56,149 - INFO - Number of databases: 11
2025-09-17 16:03:56,150 - INFO - Number of unique roles: 42
2025-09-17 16:03:56,150 - INFO - Queries denied due to permissions: 603 (33.67%)
2025-09-17 16:03:56,151 - INFO - Output saved to: /home/feiy/Role-SQL-benchmark/outputs/role_sql_dataset_20250917_160338.json


In [14]:
# # Sample some examples from the dataset
# def print_example(example):
#     print(f"Database: {example['db_id']}")
#     print(f"Role: {example['role']}")
#     print(f"Tables accessible: {example['tables']}")
#     print(f"Question: {example['question']}")
#     print(f"Query: {example['query']}")
#     print("-" * 80)

# # Sample and print 5 random examples
# print("Sample Dataset Examples:")
# print("=" * 80)
# for example in random.sample(dataset, min(5, len(dataset))):
#     print_example(example)